# FYS4480/9480, week 46: the linked-diagram theorem, and coupled cluster

Companion notebook to the week-46 slides (November 12--13, 2026), chapter 9 section 9.15 (from "Unlinked diagrams and the linked-diagram theorem" on) and chapter 10 sections 10.1--10.7. Everything is self-contained numpy; the determinant-space machinery and the diagram evaluators are those of `BookPrograms/chapter09/diagrams.py` (week 45), the CCD solver is a forty-line version of `BookPrograms/chapter10/coupledcluster.py`.

**Contents**

1. Unlinked diagrams with bare vertices: the second-order wave operator and the third-order energy, piece by piece (pairing model and a random Hamiltonian)
2. Fourth order with normal-ordered vertices: two non-interacting copies and the factorisation of the denominators
3. The two-level benchmark: the exact $g^4/\Delta^3$ against the count of exclusion-principle-violating contractions
4. Coupled-cluster doubles for the pairing model: the solver, the table of lecture 4, size extensivity
5. Coupled cluster as a resummation: the order-by-order expansion of the CCD equations against Rayleigh-Schrödinger theory, and the two partitions

Notation: $H=H_0+H_I$, $H_0=\sum_p\varepsilon_p a^\dagger_pa_p$, $H_I=\sum_{pq}u_{pq}a^\dagger_pa_q+\frac14\sum\bar v_{pqrs}a^\dagger_pa^\dagger_qa_sa_r$; the reference fills the lowest $N$ orbitals; $R_0=Q/(\varepsilon_0-H_0)$, $D_m=\varepsilon_0-\varepsilon_m$; $W=H_I-E^{(1)}$ the normal-ordered perturbation with the one-body vertex $\tilde f_{pq}=u_{pq}+U_{pq}$.

## 0. The machinery (week 45)

In [1]:
import numpy as np
from itertools import combinations

import numpy as np


# ---------------------------------------------------------------------------
#  bit-string determinants
# ---------------------------------------------------------------------------
def popcount(x):
    return bin(x).count("1")


def annihilate(state, p):
    if not (state >> p) & 1:
        return 0, 0
    sign = -1 if popcount(state & ((1 << p) - 1)) & 1 else 1
    return sign, state ^ (1 << p)


def create(state, p):
    if (state >> p) & 1:
        return 0, 0
    sign = -1 if popcount(state & ((1 << p) - 1)) & 1 else 1
    return sign, state | (1 << p)


def apply_string(state, operators):
    """Apply [(index, dagger), ...], rightmost operator first."""
    sign = 1
    for p, dagger in reversed(operators):
        s, state = (create(state, p) if dagger else annihilate(state, p))
        if s == 0:
            return 0, 0
        sign *= s
    return sign, state


class GeneralHamiltonian:
    """H = sum_p eps_p a+_p a_p + sum_pq u_pq a+_p a_q + 1/4 sum vbar a+a+aa.

    The reference determinant fills the first n_particles orbitals; the
    orbital energies eps must therefore be ordered so that these are the
    lowest.  H_0 = sum eps_p a+_p a_p, H_I = the rest.
    """

    def __init__(self, eps, u, vbar, n_particles):
        self.eps = np.asarray(eps, dtype=float)
        self.u = np.asarray(u, dtype=float)
        self.vbar = np.asarray(vbar, dtype=float)
        self.n = len(self.eps)
        self.N = n_particles
        self.occ = list(range(n_particles))
        self.vir = list(range(n_particles, self.n))
        self.states = []
        for occ in combinations(range(self.n), n_particles):
            bits = 0
            for p in occ:
                bits |= 1 << p
            self.states.append(bits)
        self.states.sort()
        self.index = {s: i for i, s in enumerate(self.states)}
        self.dim = len(self.states)
        self.reference = (1 << n_particles) - 1
        self.ref = self.index[self.reference]
        self._build()

    # ------------------------------------------------------------------
    def _build(self):
        n, dim = self.n, self.dim
        H0 = np.zeros(dim)
        V = np.zeros((dim, dim))
        for col, state in enumerate(self.states):
            H0[col] = sum(self.eps[p] for p in range(n) if (state >> p) & 1)
            for p in range(n):
                for q in range(n):
                    if self.u[p, q] == 0.0:
                        continue
                    sign, new = apply_string(state, [(p, True), (q, False)])
                    if sign:
                        V[self.index[new], col] += sign * self.u[p, q]
            for p in range(n):
                for q in range(n):
                    for r in range(n):
                        for s in range(n):
                            v = self.vbar[p, q, r, s]
                            if v == 0.0:
                                continue
                            sign, new = apply_string(
                                state, [(p, True), (q, True), (s, False), (r, False)])
                            if sign:
                                V[self.index[new], col] += 0.25 * sign * v
        self.w = H0
        self.V = V
        self.H = np.diag(H0) + V
        self.w0 = H0[self.ref]
        D = self.w0 - H0
        self.R = np.zeros(dim)
        mask = np.arange(dim) != self.ref
        self.R[mask] = 1.0 / D[mask]
        self.D = D

    # ------------------------------------------------------------------
    def rayleigh_schrodinger(self, order=4):
        """Delta E^(1) ... Delta E^(order) by the standard recursion."""
        V, R, ref = self.V, self.R, self.ref
        psi0 = np.zeros(self.dim)
        psi0[ref] = 1.0
        energies = [float(psi0 @ (V @ psi0))]
        psis = []
        for k in range(1, order):
            previous = psis[-1] if psis else psi0
            rhs = V @ previous
            for m in range(1, k):
                rhs = rhs - energies[m - 1] * psis[k - m - 1]
            psi = R * rhs
            psis.append(psi)
            energies.append(float(psi0 @ (V @ psi)))
        return energies

    def third_order_pieces(self):
        """Principal term, renormalisation term, and <W R W R W> with the
        normal-ordered vertex W = V - V_00."""
        V, R, ref = self.V, self.R, self.ref
        v0 = V[:, ref]
        principal = float(v0 @ (R * (V @ (R * v0))))
        renorm = float(V[ref, ref] * np.sum(R * R * v0 * v0))
        W = V - V[ref, ref] * np.eye(self.dim)
        w0 = W[:, ref]
        normal = float(w0 @ (R * (W @ (R * w0))))
        return principal, renorm, normal

    def exact(self):
        return float(np.linalg.eigvalsh(self.H)[0])

    # ------------------------------------------------------------------
    def ftilde(self):
        """The normal-ordered one-body vertex u + U, U_pq = sum_j vbar_pjqj."""
        U = sum(self.vbar[:, j, :, j] for j in self.occ)
        return self.u + U




def random_hamiltonian(n=8, n_particles=4, seed=1, scale_u=0.1, scale_v=0.2):
    """A random Hermitian two-body Hamiltonian with every diagram class
    non-zero: eps increasing, u symmetric and off-diagonal, vbar
    antisymmetrised and Hermitian."""
    rng = np.random.default_rng(seed)
    eps = np.sort(rng.uniform(0.0, 1.0, n))
    eps = eps - eps[0] + np.linspace(0, 2, n)             # well separated
    u = scale_u * rng.normal(size=(n, n))
    u = 0.5 * (u + u.T)
    np.fill_diagonal(u, 0.0)
    w = scale_v * rng.normal(size=(n, n, n, n))
    # Hermitian: <pq|v|rs> = <rs|v|pq>
    w = 0.5 * (w + w.transpose(2, 3, 0, 1))
    # antisymmetrise in (p,q) and (r,s)
    vbar = w - w.transpose(1, 0, 2, 3)
    vbar = vbar - vbar.transpose(0, 1, 3, 2)
    vbar *= 0.5
    return GeneralHamiltonian(eps, u, vbar, n_particles)


def add_two_body(vbar, i, j, k, l, c):
    """Add c * a+_i a+_j a_k a_l to the interaction 1/4 sum vbar_pqrs a+_p a+_q a_s a_r,
    i.e. vbar_{ijlk} += c with its three antisymmetric partners."""
    vbar[i, j, l, k] += c
    vbar[j, i, l, k] -= c
    vbar[i, j, k, l] -= c
    vbar[j, i, k, l] += c


def pairing_hamiltonian(levels=4, particles=4, g=1.0, xi=1.0):
    """The pairing model: spin-orbital 2p = (p,+), 2p+1 = (p,-), levels
    p = 0..levels-1 with energy xi*p, interaction -g/2 sum_pq a+_{p+} a+_{p-} a_{q-} a_{q+}."""
    n = 2 * levels
    eps = np.array([xi * (o // 2) for o in range(n)], dtype=float)
    u = np.zeros((n, n))
    vbar = np.zeros((n, n, n, n))
    for p in range(levels):
        for q in range(levels):
            add_two_body(vbar, 2 * p, 2 * p + 1, 2 * q + 1, 2 * q, -0.5 * g)
    return GeneralHamiltonian(eps, u, vbar, particles)


def pairing_ph_hamiltonian(levels=4, particles=4, g=1.0, f=0.0, xi=1.0):
    """The pairing plus particle-hole model of chapter 7 (rpa.py):
    pairing as above plus
        -f/2 sum_{pqr} a+_{p+} a+_{p-} a_{q-} a_{r+}  + h.c."""
    ham = pairing_hamiltonian(levels, particles, g, xi)
    vbar = ham.vbar.copy()
    for p in range(levels):
        for q in range(levels):
            for r in range(levels):
                add_two_body(vbar, 2 * p, 2 * p + 1, 2 * q + 1, 2 * r, -0.5 * f)
                add_two_body(vbar, 2 * r, 2 * q + 1, 2 * p + 1, 2 * p, -0.5 * f)
    return GeneralHamiltonian(ham.eps, ham.u, vbar, particles)


def two_copies(ham_builder, **kwargs):
    """Two non-interacting copies of a Hamiltonian, as one Hamiltonian."""
    one = ham_builder(**kwargs)
    n, N = one.n, one.N
    # orbitals of copy 2 are shifted by n; the reference must fill the lowest
    # 2N orbitals, so interleave: copy-1 occupied, copy-2 occupied, then the rest
    order = list(range(N)) + list(range(n, n + N)) + list(range(N, n)) + list(range(n + N, 2 * n))
    eps2 = np.concatenate([one.eps, one.eps])[order]
    u2 = np.zeros((2 * n, 2 * n)); v2 = np.zeros((2 * n,) * 4)
    u_full = np.zeros((2 * n, 2 * n)); u_full[:n, :n] = one.u; u_full[n:, n:] = one.u
    v_full = np.zeros((2 * n,) * 4); v_full[:n, :n, :n, :n] = one.vbar; v_full[n:, n:, n:, n:] = one.vbar
    u2 = u_full[np.ix_(order, order)]
    v2 = v_full[np.ix_(order, order, order, order)]
    return one, GeneralHamiltonian(eps2, u2, v2, 2 * N)



def fourth_order_pieces(ham):
    """E4 = principal - E1[mixed] + E1^2 cubed - E2 folded, in resolvent form."""
    V, R, ref = ham.V, ham.R, ham.ref
    v0 = V[:, ref]
    e1 = V[ref, ref]
    e2 = float(np.sum(R * v0 * v0))
    r1 = R * v0                       # R V |0>
    principal = float(v0 @ (R * (V @ (R * (V @ r1)))))
    mixed = float(v0 @ (R * R * (V @ r1))) + float(v0 @ (R * (V @ (R * r1))))
    cubed = float(np.sum(R ** 3 * v0 * v0))
    folded = float(np.sum(R * R * v0 * v0))
    return principal - e1 * mixed + e1 ** 2 * cubed - e2 * folded, principal, mixed, cubed, folded


In [2]:
# ---------------------------------------------------------------------------
#  the diagrams
# ---------------------------------------------------------------------------
def second_order_diagrams(ham):
    """The two second-order diagrams with normal-ordered vertices."""
    eps, vb = ham.eps, ham.vbar
    o, v = ham.occ, ham.vir
    f = ham.ftilde()
    e_o, e_v = eps[o], eps[v]
    Dai = e_o[:, None] - e_v[None, :]                       # [i, a]
    fov = f[np.ix_(o, v)]                                  # f_ia
    fvo = f[np.ix_(v, o)]                                  # f_ai
    singles = float(np.sum(fov * fvo.T / Dai))
    Dijab = (e_o[:, None, None, None] + e_o[None, :, None, None]
             - e_v[None, None, :, None] - e_v[None, None, None, :])
    v_oovv = vb[np.ix_(o, o, v, v)]
    v_vvoo = vb[np.ix_(v, v, o, o)]
    doubles = 0.25 * float(np.einsum("ijab,abij,ijab->", v_oovv, v_vvoo, 1.0 / Dijab))
    return {"(F,F) singles": singles, "(V,V) doubles": doubles}


def third_order_diagrams(ham):
    """The fourteen connected third-order energy diagrams.

    Vertices are read from the bottom (first to act) to the top, F is the
    normal-ordered one-body vertex ftilde and V the interaction.  Each value
    is obtained with the rules: sums over labels, one denominator per cut,
    1/2 per pair of equivalent lines, sign (-1)^(h+l).
    """
    eps, vb = ham.eps, ham.vbar
    o, v = ham.occ, ham.vir
    f = ham.ftilde()
    e_o, e_v = eps[o], eps[v]
    foo = f[np.ix_(o, o)]       # f_ij
    fvv = f[np.ix_(v, v)]       # f_ab
    fov = f[np.ix_(o, v)]       # f_ia
    fvo = f[np.ix_(v, o)]       # f_ai
    D1 = e_o[:, None] - e_v[None, :]                               # D[i,a]
    D2 = (e_o[:, None, None, None] + e_o[None, :, None, None]
          - e_v[None, None, :, None] - e_v[None, None, None, :])   # D[i,j,a,b]
    v_oovv = vb[np.ix_(o, o, v, v)]   # <ij|v|ab>
    v_vvoo = vb[np.ix_(v, v, o, o)]   # <ab|v|ij>
    v_ovov = vb[np.ix_(o, v, o, v)]   # <ib|v|ja>
    v_vvvo = vb[np.ix_(v, v, v, o)]   # <bc|v|aj>
    v_ovoo = vb[np.ix_(o, v, o, o)]   # <ib|v|jk>
    v_vvvv = vb[np.ix_(v, v, v, v)]   # <cd|v|ab>
    v_oooo = vb[np.ix_(o, o, o, o)]   # <ij|v|kl>
    v_vovo = vb[np.ix_(v, o, v, o)]   # <ci|v|ak>
    v_ovvv = vb[np.ix_(o, v, v, v)]   # <ic|v|ab>
    v_oovo = vb[np.ix_(o, o, v, o)]   # <ij|v|ak>

    d = {}
    # (F,F,F)
    d["1  (F,F,F) particle insertion"] = float(np.einsum(
        "ib,ba,ai,ia,ib->", fov, fvv, fvo, 1 / D1, 1 / D1))
    d["2  (F,F,F) hole insertion"] = -float(np.einsum(
        "ja,ij,ai,ia,ja->", fov, foo, fvo, 1 / D1, 1 / D1))
    # (F,F,V)
    d["3  (F,F,V) two crosses closed by V"] = float(np.einsum(
        "ijab,bj,ai,ia,ijab->", v_oovv, fvo, fvo, 1 / D1, 1 / D2))
    # (F,V,F)
    d["4  (F,V,F) particle-hole scattering"] = -float(np.einsum(
        "jb,ibja,ai,ia,jb->", fov, v_ovov, fvo, 1 / D1, 1 / D1))
    # (F,V,V)
    d["5  (F,V,V) particle line into V"] = 0.5 * float(np.einsum(
        "ijbc,bcaj,ai,ia,ijbc->", v_oovv, v_vvvo, fvo, 1 / D1, 1 / D2))
    d["6  (F,V,V) hole line into V"] = -0.5 * float(np.einsum(
        "jkab,ibjk,ai,ia,jkab->", v_oovv, v_ovoo, fvo, 1 / D1, 1 / D2))
    # (V,F,V)
    d["7  (V,F,V) particle insertion"] = 0.5 * float(np.einsum(
        "ijcb,ca,abij,ijab,ijcb->", v_oovv, fvv, v_vvoo, 1 / D2, 1 / D2))
    d["8  (V,F,V) hole insertion"] = -0.5 * float(np.einsum(
        "kjab,ik,abij,ijab,kjab->", v_oovv, foo, v_vvoo, 1 / D2, 1 / D2))
    # (V,F,F)
    d["9  (V,F,F) V closed by two crosses"] = float(np.einsum(
        "jb,ia,abij,ijab,jb->", fov, fov, v_vvoo, 1 / D2, 1 / D1))
    # (V,V,V)
    d["10 (V,V,V) particle-particle ladder"] = 0.125 * float(np.einsum(
        "ijcd,cdab,abij,ijab,ijcd->", v_oovv, v_vvvv, v_vvoo, 1 / D2, 1 / D2))
    d["11 (V,V,V) hole-hole ladder"] = 0.125 * float(np.einsum(
        "klab,ijkl,abij,ijab,klab->", v_oovv, v_oooo, v_vvoo, 1 / D2, 1 / D2))
    d["12 (V,V,V) particle-hole ring"] = -float(np.einsum(
        "kjcb,ciak,abij,ijab,kjcb->", v_oovv, v_vovo, v_vvoo, 1 / D2, 1 / D2))
    # (V,V,F)
    d["13 (V,V,F) V to single via particles"] = 0.5 * float(np.einsum(
        "jc,icab,abij,ijab,jc->", fov, v_ovvv, v_vvoo, 1 / D2, 1 / D1))
    d["14 (V,V,F) V to single via holes"] = -0.5 * float(np.einsum(
        "kb,ijak,abij,ijab,kb->", fov, v_oovo, v_vvoo, 1 / D2, 1 / D1))
    return d




## 1. Unlinked diagrams with bare vertices

With bare vertices $V=H_I$ the second-order wave operator is $\Omega^{(2)}=RVRV-E^{(1)}R^2V$. In $\langle\Phi_m|RVRV|\Phi_0\rangle=\sum_n V_{mn}V_{n0}/(D_mD_n)$ the diagonal term $n=m$ contains $V_{mm}=V_{00}+W_{mm}$, and the $V_{00}$ part is the **unlinked** drawing: the closed first-order piece ($u$ loop, two bubbles) floating beside the open first-order piece, worth $V_{00}V_{m0}/D_m^2$. The renormalisation term $-E^{(1)}R^2V$ is exactly its negative. Closing with a top vertex gives the third-order energy: principal term $=$ connected diagrams $+$ unlinked diagram $E^{(1)}\langle VR^2V\rangle$, and the renormalisation term removes the latter.

For the pairing model the connected part is itself three diagrams (pp ladder, hh ladder, bubbles on the two hole lines), and the bubble insertions happen to cancel the unlinked diagram exactly ($W_{mm}=0$), which is why the principal term looks like the ladders alone.

In [3]:
def unlinked_pieces(ham, title):
    V, R, ref = ham.V, ham.R, ham.ref
    v0 = V[:, ref]
    V00 = V[ref, ref]
    # second-order wave operator
    psi2_principal = R * (V @ (R * v0))
    psi2_unlinked = V00 * R * R * v0                      # the closed one-vertex piece beside the open piece
    psi2_recursion = ham.rayleigh_schrodinger(4)           # energies only; rebuild the state:
    psi2 = psi2_principal - psi2_unlinked
    # the diagonal of the middle vertex, V_mm = V_00 + W_mm
    diag = np.diag(V)
    Wmm = diag - V00
    # third-order energy
    principal = float(v0 @ psi2_principal)
    unlinked = float(V00 * np.sum(R * R * v0 * v0))        # E1 <V R^2 V>
    e3 = ham.rayleigh_schrodinger(3)[2]
    print("=" * 78); print(title); print("=" * 78)
    print(f"   E1 = V00                         {V00:16.10f}")
    print(f"   <V R^2 V>                        {np.sum(R*R*v0*v0):16.10f}")
    print(f"   principal <V R V R V>            {principal:16.10f}")
    print(f"   unlinked  E1 <V R^2 V>           {unlinked:16.10f}")
    print(f"   principal - unlinked             {principal-unlinked:16.10f}   = E3 from the recursion {e3:16.10f}")
    print(f"   max |W_mm| over the excited states {np.max(np.abs(Wmm[np.arange(ham.dim)!=ref])):.3e}")
    return principal, unlinked

p, u = unlinked_pieces(pairing_hamiltonian(g=0.5), "pairing model, four levels, two pairs, g = 0.5")
d = third_order_diagrams(pairing_hamiltonian(g=0.5))
pp, hh = d["10 (V,V,V) particle-particle ladder"], d["11 (V,V,V) hole-hole ladder"]
ins = d["8  (V,F,V) hole insertion"]
print(f"   pp ladder {pp:.10f}  hh ladder {hh:.10f}  hole bubbles {ins:.10f}  unlinked {u:.10f}")
print(f"   sum of the four families {pp+hh+ins+u:.10f}   = bare principal term {p:.10f}")
print(f"   connected part (ladders + bubbles) {pp+hh+ins:.10f}")
unlinked_pieces(random_hamiltonian(), "random Hamiltonian, n = 8, N = 4 (W_mm is not zero here)");

pairing model, four levels, two pairs, g = 0.5
   E1 = V00                            -0.5000000000
   <V R^2 V>                            0.0251736111
   principal <V R V R V>               -0.0230034722
   unlinked  E1 <V R^2 V>              -0.0125868056
   principal - unlinked                -0.0104166667   = E3 from the recursion    -0.0104166667
   max |W_mm| over the excited states 5.000e-01
   pp ladder -0.0115017361  hh ladder -0.0115017361  hole bubbles 0.0125868056  unlinked -0.0125868056
   sum of the four families -0.0230034722   = bare principal term -0.0230034722
   connected part (ladders + bubbles) -0.0104166667
random Hamiltonian, n = 8, N = 4 (W_mm is not zero here)
   E1 = V00                            -0.4711329911
   <V R^2 V>                            1.1492160219
   principal <V R V R V>               -0.1417616190
   unlinked  E1 <V R^2 V>              -0.5414335818
   principal - unlinked                 0.3996719628   = E3 from the recursion     0.39967196

## 2. Fourth order with normal-ordered vertices: two non-interacting copies

With $W=H_I-E^{(1)}$ the one-vertex closed pieces are gone and $E^{(4)}=\langle WR_0WR_0WR_0W\rangle-E^{(2)}\langle WR_0^2W\rangle$. Two non-interacting copies $A$ and $B$ of a three-level pairing system with one pair each (no shared orbitals, hence no exclusion-principle-violating terms). Restricting the intermediate states of the principal term to the four patterns $(A,AB,B)$, $(B,AB,A)$, $(A,AB,A)$, $(B,AB,B)$ isolates the unlinked diagrams; by the factorisation of the denominators,
$$\frac{1}{D_m(D_m+D_n)D_n}+\frac{1}{D_m(D_m+D_n)D_m}=\frac{1}{D_m^2D_n},$$
their sum must equal the cross-copy part of the folded term, $2E^{(2)}_A\langle WR_0^2W\rangle_A$, and $E^{(4)}_{AB}=2E^{(4)}_A$.

In [4]:
def fourth_order_unlinked(builder, **kwargs):
    """Two non-interacting copies A and B.  With normal-ordered vertices W,
    E4 = <WRWRWRW> - E2 <WR^2W>.  The part of the principal term in which the
    intermediate states visit A, then A and B together, then B (or A, AB, A,
    and the two with the roles exchanged) is the sum of the unlinked
    fourth-order diagrams; it must equal the cross-copy part of the
    renormalisation term, 2 E2(one copy) <WR^2W>(one copy)."""
    one, two = two_copies(builder, **kwargs)
    n, N = one.n, one.N
    def excited(state):
        a = sum(1 for p in range(2 * N, N + n) if (state >> p) & 1)
        b = sum(1 for p in range(N + n, 2 * n) if (state >> p) & 1)
        return a > 0, b > 0
    flags = [excited(s) for s in two.states]
    PA = np.array([a and not b for a, b in flags], float)
    PB = np.array([b and not a for a, b in flags], float)
    PAB = np.array([a and b for a, b in flags], float)
    W = two.V - two.V[two.ref, two.ref] * np.eye(two.dim)
    R = two.R
    w0 = W[:, two.ref]
    def chain(P1, P2, P3):
        x = R * P1 * w0
        x = R * P2 * (W @ x)
        x = R * P3 * (W @ x)
        return float(w0 @ x)
    unlinked = chain(PA, PAB, PB) + chain(PB, PAB, PA) + chain(PA, PAB, PA) + chain(PB, PAB, PB)
    W1 = one.V - one.V[one.ref, one.ref] * np.eye(one.dim)
    w1 = W1[:, one.ref]
    e2_one = float(np.sum(one.R * w1 * w1))
    r2_one = float(np.sum(one.R ** 2 * w1 * w1))
    e_one = one.rayleigh_schrodinger(4)
    e_two = two.rayleigh_schrodinger(4)
    return {"unlinked diagrams (four time orderings)": unlinked,
            "2 E2(A) <WR^2W>(A)": 2 * e2_one * r2_one,
            "E4 two copies": e_two[3], "2 x E4 one copy": 2 * e_one[3],
            "A,AB,B": chain(PA, PAB, PB), "A,AB,A": chain(PA, PAB, PA)}

res = fourth_order_unlinked(pairing_hamiltonian, levels=3, particles=2, g=0.5)
for k, v in res.items():
    print(f"   {k:42s} {v:18.14f}")
print("   the (A,AB,B) and (A,AB,A) orderings alone are not products; only their sum factorises")

   unlinked diagrams (four time orderings)     -0.00183105468750
   2 E2(A) <WR^2W>(A)                          -0.00183105468750
   E4 two copies                                0.00109863281250
   2 x E4 one copy                              0.00109863281250
   A,AB,B                                      -0.00043741861979
   A,AB,A                                      -0.00047810872396
   the (A,AB,B) and (A,AB,A) orderings alone are not products; only their sum factorises


## 3. The two-level benchmark: $g^4/\Delta^3$ from exclusion-principle-violating diagrams

One particle in two orbitals with the off-diagonal coupling $g$ and the gap $\Delta$: $E_--\varepsilon_1=-g^2/\Delta+g^4/\Delta^3+\mathcal O(g^6)$. In the determinant calculation the whole fourth order is the folded term, $-E^{(2)}\langle WR^2W\rangle=-g^4/D^3$. In the diagram calculation the four complete contractions of the four-cross diagram have the value $F=g^4/(2D^3)$ each before the sign: the two with the same matching for particles and holes are unlinked ($+2F$), the two with opposite matchings are linked ($-2F$); the principal term is $0$, the folded term removes $+2F$, and $E^{(4)}=-2F=-g^4/D^3=+g^4/\Delta^3$ comes entirely from linked diagrams that violate the exclusion principle at the middle cut.

In [5]:
Delta = 1.0
for g in (0.05, 0.1, 0.2):
    D = -Delta
    exact = (Delta - np.sqrt(Delta**2 + 4 * g**2)) / 2        # E_- - eps_1
    fourth = exact + g**2 / Delta                              # remove the second order
    F = g**4 / (2 * D**3)
    print(f"g = {g:4.2f}:  exact - E2 = {fourth: .10f}   g^4/Delta^3 = {g**4/Delta**3: .10f}   "
          f"-2F = {-2*F: .10f}   principal 2F-2F = {2*F-2*F: .1f}   (difference is O(g^6): {fourth - g**4/Delta**3: .2e})")

g = 0.05:  exact - E2 =  0.0000062189   g^4/Delta^3 =  0.0000062500   -2F =  0.0000062500   principal 2F-2F =  0.0   (difference is O(g^6): -3.11e-08)
g = 0.10:  exact - E2 =  0.0000980486   g^4/Delta^3 =  0.0001000000   -2F =  0.0001000000   principal 2F-2F =  0.0   (difference is O(g^6): -1.95e-06)
g = 0.20:  exact - E2 =  0.0014835193   g^4/Delta^3 =  0.0016000000   -2F =  0.0016000000   principal 2F-2F =  0.0   (difference is O(g^6): -1.16e-04)


## 4. Coupled-cluster doubles for the pairing model

A general spin-orbital CCD solver, forty lines of `einsum`, in the Møller-Plesset partition (denominators from the diagonal of the Fock matrix, $\varepsilon_i-g/2$ for the occupied levels of the pairing model). The amplitude equation is that of lecture 4 (chapter 10, eq. for CCD): driving term, two Fock terms, pp ladder, hh ladder, ring, and the four quadratic terms; iteration starts from the first-order amplitude $\langle ab\Vert ij\rangle/D_{ij}^{ab}$.

In [6]:


def ccd(ham, tol=1e-12, maxiter=200, orders=None, verbose=False):
    """Coupled-cluster doubles in the spin-orbital basis of a GeneralHamiltonian.
    Moeller-Plesset partition: denominators from the diagonal of the Fock matrix.
    Returns (correlation energy, amplitudes, iterations)."""
    o, v = ham.occ, ham.vir
    f = np.diag(ham.eps) + ham.u + sum(ham.vbar[:, j, :, j] for j in o)
    vb = ham.vbar
    foo, fvv = f[np.ix_(o, o)], f[np.ix_(v, v)]
    fp_oo = foo - np.diag(np.diag(foo)); fp_vv = fvv - np.diag(np.diag(fvv))
    eo, ev = np.diag(foo), np.diag(fvv)
    D = eo[:, None, None, None] + eo[None, :, None, None] - ev[None, None, :, None] - ev[None, None, None, :]
    v_vvoo = vb[np.ix_(v, v, o, o)]; v_oovv = vb[np.ix_(o, o, v, v)]
    v_vvvv = vb[np.ix_(v, v, v, v)]; v_oooo = vb[np.ix_(o, o, o, o)]
    v_ovvo = vb[np.ix_(o, v, v, o)]          # <kb|v|cj>
    drive = v_vvoo.transpose(2, 3, 0, 1)     # <ab|v|ij> as [i,j,a,b]

    def Pij(x): return x - x.transpose(1, 0, 2, 3)
    def Pab(x): return x - x.transpose(0, 1, 3, 2)

    def rhs(t):
        r = drive.copy()
        r += Pab(np.einsum("bc,ijac->ijab", fp_vv, t))
        r -= Pij(np.einsum("kj,ikab->ijab", fp_oo, t))
        r += 0.5 * np.einsum("abcd,ijcd->ijab", v_vvvv, t)
        r += 0.5 * np.einsum("klij,klab->ijab", v_oooo, t)
        r += Pij(Pab(np.einsum("kbcj,ikac->ijab", v_ovvo, t)))
        r += 0.25 * np.einsum("klcd,ijcd,klab->ijab", v_oovv, t, t)
        r += 0.5 * Pij(Pab(np.einsum("klcd,ikac,jlbd->ijab", v_oovv, t, t)))
        r -= 0.5 * Pij(np.einsum("klcd,ikab,jlcd->ijab", v_oovv, t, t))
        r -= 0.5 * Pab(np.einsum("klcd,ijac,klbd->ijab", v_oovv, t, t))
        return r

    def energy(t): return 0.25 * np.einsum("ijab,ijab->", v_oovv, t)

    t = drive / D
    e = energy(t)
    for it in range(1, maxiter + 1):
        t_new = rhs(t) / D
        e_new = energy(t_new)
        if verbose: print(it, e_new)
        conv = abs(e_new - e) < tol
        t, e = t_new, e_new
        if conv: break
    return e, t, it


def ccd_orders(ham, nmax=8):
    """Order-by-order expansion of the CCD equations, D t^(k) = L[t^(k-1)] + sum Q[t^(i),t^(j)]."""
    o, v = ham.occ, ham.vir
    f = np.diag(ham.eps) + ham.u + sum(ham.vbar[:, j, :, j] for j in o)
    vb = ham.vbar
    foo, fvv = f[np.ix_(o, o)], f[np.ix_(v, v)]
    fp_oo = foo - np.diag(np.diag(foo)); fp_vv = fvv - np.diag(np.diag(fvv))
    eo, ev = np.diag(foo), np.diag(fvv)
    D = eo[:, None, None, None] + eo[None, :, None, None] - ev[None, None, :, None] - ev[None, None, None, :]
    v_vvoo = vb[np.ix_(v, v, o, o)]; v_oovv = vb[np.ix_(o, o, v, v)]
    v_vvvv = vb[np.ix_(v, v, v, v)]; v_oooo = vb[np.ix_(o, o, o, o)]
    v_ovvo = vb[np.ix_(o, v, v, o)]
    drive = v_vvoo.transpose(2, 3, 0, 1)
    def Pij(x): return x - x.transpose(1, 0, 2, 3)
    def Pab(x): return x - x.transpose(0, 1, 3, 2)
    def L(t):
        r = Pab(np.einsum("bc,ijac->ijab", fp_vv, t)) - Pij(np.einsum("kj,ikab->ijab", fp_oo, t))
        r += 0.5 * np.einsum("abcd,ijcd->ijab", v_vvvv, t) + 0.5 * np.einsum("klij,klab->ijab", v_oooo, t)
        r += Pij(Pab(np.einsum("kbcj,ikac->ijab", v_ovvo, t)))
        return r
    def Q(s, t):   # symmetric bilinear form
        r = 0.25 * np.einsum("klcd,ijcd,klab->ijab", v_oovv, s, t)
        r += 0.5 * Pij(Pab(np.einsum("klcd,ikac,jlbd->ijab", v_oovv, s, t)))
        r -= 0.5 * Pij(np.einsum("klcd,ikab,jlcd->ijab", v_oovv, s, t))
        r -= 0.5 * Pab(np.einsum("klcd,ijac,klbd->ijab", v_oovv, s, t))
        return 0.5 * (r + (0.25 * np.einsum("klcd,ijcd,klab->ijab", v_oovv, t, s)
                       + 0.5 * Pij(Pab(np.einsum("klcd,ikac,jlbd->ijab", v_oovv, t, s)))
                       - 0.5 * Pij(np.einsum("klcd,ikab,jlcd->ijab", v_oovv, t, s))
                       - 0.5 * Pab(np.einsum("klcd,ijac,klbd->ijab", v_oovv, t, s))))
    ts = [drive / D]
    for k in range(2, nmax + 1):
        r = L(ts[-1])
        for i in range(1, k - 1):
            r = r + Q(ts[i - 1], ts[k - 2 - i])
        ts.append(r / D)
    return [0.25 * np.einsum("ijab,ijab->", v_oovv, t) for t in ts]   # E^(k+1)




In [7]:
print("   g      E_ref        CCD           FCI          error     iterations")
for g in (0.25, 0.5, 1.0, 1.5, 2.0):
    ham = pairing_hamiltonian(g=g)
    e, t, it = ccd(ham)
    print(f"  {g:4.2f}  {2-g:8.5f}  {2-g+e:12.8f}  {ham.exact():12.8f}  {2-g+e-ham.exact():10.2e}   {it}")
# size extensivity: two and three copies of a three-level system with one pair
print("\n   size extensivity, three-level pairing system with one pair:")
for g in (0.25, 0.5, 1.0):
    one, two = two_copies(pairing_hamiltonian, levels=3, particles=2, g=g)
    e1 = ccd(one)[0]; e2 = ccd(two)[0]
    print(f"   g = {g:4.2f}   E(1) = {e1:16.12f}   E(2) - 2 E(1) = {e2 - 2*e1: .2e}")

   g      E_ref        CCD           FCI          error     iterations
  0.25   1.75000    1.73045621    1.73045985   -3.63e-06   13
  0.50   1.50000    1.41663766    1.41677428   -1.37e-04   18
  1.00   1.00000    0.63044275    0.63554847   -5.11e-03   26


  1.50   0.50000   -0.38416526   -0.34819051   -3.60e-02   30


  2.00   0.00000   -1.60959440   -1.48965216   -1.20e-01   31

   size extensivity, three-level pairing system with one pair:


   g = 0.25   E(1) =  -0.012167069342   E(2) - 2 E(1) =  0.00e+00


   g = 0.50   E(1) =  -0.050059379847   E(2) - 2 E(1) = -2.11e-13


   g = 1.00   E(1) =  -0.205303400091   E(2) - 2 E(1) =  0.00e+00


## 5. Coupled cluster as a resummation of perturbation theory

Write the CCD equation as $D\,t=C+L[t]+Q[t,t]$ and expand $t=\sum_kt^{(k)}$: $Dt^{(1)}=C$, $Dt^{(k)}=L[t^{(k-1)}]+\sum_{i+j=k-1}Q[t^{(i)},t^{(j)}]$, with $E^{(k+1)}=\frac14\sum\langle ij\Vert ab\rangle t^{(k)}{}_{ij}^{ab}$. These are the Møller-Plesset energies in the partition with the Fock operator in $H_0$; compare with the determinant-space Rayleigh-Schrödinger recursion of week 44 in the *same* partition (move $U_{pq}$ into the orbital energies, i.e. shift the occupied levels by $-g/2$), and with the one-body partition of week 45 ($-\frac7{24}g^2$ at second order).

In [8]:
def pairing_mp_partition(levels=4, particles=4, g=1.0, xi=1.0):
    """The pairing model with the Fock operator in H_0: occupied levels shifted by -g/2,
    and the compensating one-body term -U_pq in H_I (so H is unchanged)."""
    ham = pairing_hamiltonian(levels, particles, g, xi)
    U = sum(ham.vbar[:, j, :, j] for j in ham.occ)
    eps = ham.eps + np.diag(U)
    u = ham.u - U
    return GeneralHamiltonian(eps, u, ham.vbar, particles)

for g in (0.5, 1.0):
    es = ccd_orders(pairing_hamiltonian(g=g), nmax=16)
    rs = pairing_mp_partition(g=g).rayleigh_schrodinger(5)
    print(f"g = {g}")
    for k in range(3):
        print(f"   order {k+2}:  from CCD {es[k]: .10f}   from MBPT (MP partition) {rs[k+1]: .10f}")
    e, t, it = ccd(pairing_hamiltonian(g=g))
    print(f"   sum of 16 orders {sum(es): .7f}   CCD solved directly {e: .7f}")
    bare = pairing_hamiltonian(g=g).rayleigh_schrodinger(3)
    print(f"   second order, one-body partition (week 45): {bare[1]: .7f} = -7/24 g^2 = {-7/24*g**2: .7f};"
          f"  MP partition: {-(g**2/4)*(1/(2+g)+2/(4+g)+1/(6+g)): .7f}")

g = 0.5
   order 2:  from CCD -0.0623931624   from MBPT (MP partition) -0.0623931624
   order 3:  from CCD -0.0165183724   from MBPT (MP partition) -0.0165183724
   order 4:  from CCD -0.0038210766   from MBPT (MP partition) -0.0038210766
   sum of 16 orders -0.0833623   CCD solved directly -0.0833623
   second order, one-body partition (week 45): -0.0729167 = -7/24 g^2 = -0.0729167;  MP partition: -0.0623932


g = 1.0
   order 2:  from CCD -0.2190476190   from MBPT (MP partition) -0.2190476190
   order 3:  from CCD -0.1004988662   from MBPT (MP partition) -0.1004988662
   order 4:  from CCD -0.0403096858   from MBPT (MP partition) -0.0403096858
   sum of 16 orders -0.3695594   CCD solved directly -0.3695572
   second order, one-body partition (week 45): -0.2916667 = -7/24 g^2 = -0.2916667;  MP partition: -0.2190476
